In [59]:
import sys
from pathlib import Path

import numpy as np
import open3d as o3d

ROOT_DIR = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "src").is_dir() and (directory / "data").is_dir()
)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

PROCESSED_DIR = ROOT_DIR / "data/processed"
REPORT_IMAGES_DIR = ROOT_DIR / "report/report_images"

from src.utils.plotting import show_geometries, save_open3d_screenshot

## Paso 0 - cargar nube global

In [60]:
ply_path = PROCESSED_DIR / "palacio" / "nube_global_palacio.ply"
pcd = o3d.io.read_point_cloud(str(ply_path))

n_inicial = len(pcd.points)
print(f"Puntos iniciales: {n_inicial:,}")

# registro de puntos sobrevivientes por etapa (para el informe)
conteos = {"inicial": n_inicial}

show_geometries([pcd])

Puntos iniciales: 87,150,016


## Paso 1 - Filtrado espacial (recorte de caja)

In [61]:
puntos = np.asarray(pcd.points)
print("min (x,y,z):", puntos.min(axis=0))
print("max (x,y,z):", puntos.max(axis=0))

min (x,y,z): [-3417.86434274 -3420.34731459 -1004.69847093]
max (x,y,z): [3583.41896217 3557.01828882 2608.55673503]


In [62]:
min_bound = np.array([130.0, 90.0, -140.0])  
max_bound = np.array([ 220.0,  470.0,  0.0])    
bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
bbox.color = (1, 0, 0)

pcd_bbox = pcd.crop(bbox)

conteos["bbox"] = len(pcd_bbox.points)
print(f"Antes:   {len(pcd.points):,}")
print(f"Después: {conteos['bbox']:,}")

show_geometries([pcd_bbox])

Antes:   87,150,016
Después: 8,901,754


## Paso 2 - Submuestreo (voxel donwsample)

In [63]:
voxel_size = 3.0   # mm — parámetro a ajustar
pcd_voxel = pcd_bbox.voxel_down_sample(voxel_size=voxel_size)

conteos["voxel"] = len(pcd_voxel.points)
print(f"Tras voxel ({voxel_size} mm): {conteos['voxel']:,} puntos")
show_geometries([pcd_voxel])

Tras voxel (3.0 mm): 73,187 puntos


## Paso 3 - Eliminación de Outiliers

In [70]:
pcd_limpio, idx = pcd_voxel.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=1.5,
)

conteos["outliers"] = len(pcd_limpio.points)
print(f"Tras outliers: {conteos['outliers']:,} puntos")
show_geometries([pcd_limpio])

Tras outliers: 69,093 puntos


In [69]:
outliers = pcd_voxel.select_by_index(idx, invert=True)   # los que NO sobrevivieron
outliers.paint_uniform_color([1, 0, 0])
show_geometries([pcd_limpio, outliers])